In [0]:
from olist_lakehouse.io import read_managed_table, write_managed_table
from olist_lakehouse.transformations import parse_timestamp
from pyspark.sql.functions import col, timestamp_diff, trim, when

In [0]:
dbutils.widgets.text("catalog", "")

In [0]:
catalog = dbutils.widgets.get("catalog")

## Reviews table
Enriched reviews table with sentiment analysis and response metrics.

**Table requirements:**

- Create a `sentiment_bucket`:
  - `'negative'` → scores 1 and 2
  - `'neutral'` → score 3
  - `'positive'` → scores 4 and 5
- `has_comment` = `TRUE` when `review_comment_message` is not null and not empty
- `response_time_hours` = `review_creation_date` - `review_answer_timestamp`

**Expected Schema:**

| Field                   | Type      | Origin (bronze)             |
| ------------------------| ----------| ----------------------------|
| review_id               | STRING    | olist_order_reviews_dataset |
| order_id                | STRING    | olist_order_reviews_dataset |
| review_score            | INT       | olist_order_reviews_dataset |
| review_comment_title    | STRING    | olist_order_reviews_dataset |
| review_comment_message  | STRING    | olist_order_reviews_dataset |
| review_creation_date    | TIMESTAMP | olist_order_reviews_dataset |
| review_answer_timestamp | TIMESTAMP | olist_order_reviews_dataset |
| sentiment_bucket        | STRING    | logic                       |
| has_comment             | BOOLEAN   | logic                       |
| response_time_hours     | BIGINT    | logic                       |

In [0]:
reviews = read_managed_table(
    catalog=catalog,
    schema="bronze",
    table="olist_order_reviews_dataset",
)

In [0]:
reviews_select = reviews.select(
    "review_id",
    "order_id",
    col("review_score").cast("int"),
    "review_comment_title",
    "review_comment_message",
    parse_timestamp(col("review_creation_date")).alias("review_creation_date"),
    parse_timestamp(col("review_answer_timestamp")).alias("review_answer_timestamp"),
)

reviews_transformed = (
    reviews_select.withColumn(
        "sentiment_bucket",
        when(col("review_score").isin([1, 2]), "negative")
        .when(col("review_score") == 3, "neutral")
        .when(col("review_score").isin([4, 5]), "positive")
        .otherwise(None),
    )
    .withColumn(
        "has_comment",
        when(trim(col("review_comment_message")).isNull(), False).otherwise(True),
    )
    .withColumn(
        "response_time_hours",
        timestamp_diff("HOUR", col("review_creation_date"), col("review_answer_timestamp")),
    )
)

In [0]:
write_managed_table(
    reviews_transformed,
    mode="overwrite",
    table_full_path=f"{catalog}.silver.reviews",
    options={"overwriteSchema": "true"}
)